Soil Agent: Kenya Soil DSS prototype

What it is

An AI agent that answers questions about soil conditions and gives management recommendations, using the soil maps produced by the workflow in notebook "Quantile_Forest_Regression" under notebooks. It is built with smolagents from Hugging Face, as a CodeAgent powered by Qwen2.5-Coder-32B-Instruct through Hugging Face Inference.

What it does

Answers questions in plain language, such as "What is the soil pH around Kitale?", "Should a farmer near Kakamega town apply lime?" or "How much of the area within 5 km of Malava has low organic carbon?"
Finds places by name, looks up soil conditions at a point or across an area, and explains what the values mean.
Gives recommendations based on the soil class and its interpretation, stating the confidence and whether the location is within the area the model represents well.

Current coverage

Area: Kakamega and Trans Nzoia counties
Properties: soil pH and soil organic carbon (SOC), 0–30 cm
Data: legacy GloSIS-harmonized soil data, Quantile Regression Forest predictions, and indicator classes from config/indicator_thresholds.csv

Author

Japheth Kimeu
Geospatial & AI Technical Lead
FAO Kenya, SoilFER project

Version: prototype v0.1, September 2026

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/drive/MyDrive/soil_dss'   # adjust if your repo folder is elsewhere
os.chdir(REPO)
sys.path.insert(0, REPO)
print(os.listdir('agent'), os.listdir('outputs'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['soil_tools.py', '__init__.py', '__pycache__', 'soil_agent.py'] ['model', 'maps', 'indicators']


In [2]:
!pip install -q smolagents rasterio

In [3]:
import json
from agent.soil_tools import find_place, get_soil_at_location, summarize_area, get_class_definitions

print(json.dumps(find_place(name="Kitale"), indent=1))
print(json.dumps(get_soil_at_location(lat=0.28, lon=34.75), indent=1))
print(json.dumps(summarize_area(lat=1.02, lon=35.0, radius_km=5), indent=1))
print(json.dumps(get_class_definitions(property_name="ph"), indent=1))

{
 "query": "Kitale",
 "matches": [
  {
   "name": "Kitale, Saboti, Trans Nzoia, Kenya",
   "lat": 1.01808,
   "lon": 35.00024,
   "type": "town"
  }
 ]
}
{
 "location": {
  "lat": 0.28,
  "lon": 34.75
 },
 "depth": "0-30 cm",
 "properties": {
  "ph": {
   "summary": "Soil pH (water): 5.0 pH units - Very strongly acid, possibly strongly acid (medium confidence); likely range 4.8-5.3 pH units; inside the area well represented by the sampled sites.",
   "label": "Soil pH (water)",
   "unit": "pH units",
   "predicted_value": 5.0,
   "likely_range_50pct": [
    4.8,
    5.3
   ],
   "range_90pct": [
    4.2,
    5.8
   ],
   "class": "Very strongly acid",
   "class_range": "4.5 to <5.1 pH units",
   "possible_classes": [
    "Very strongly acid",
    "Strongly acid"
   ],
   "confidence": "medium",
   "inside_area_of_applicability": true,
   "interpretation": "Aluminium toxicity likely and P poorly available; liming strongly advised",
   "explanation": "Aluminium toxicity commonly restric

In [4]:
import os
print(os.listdir('agent'))

['soil_tools.py', '__init__.py', '__pycache__', 'soil_agent.py']


In [6]:
from google.colab import userdata
from huggingface_hub import whoami
from agent.soil_agent import build_agent

token = userdata.get('HF_TOKEN')
print(whoami(token=token)['name'])      # should print: kimeujapheth

agent = build_agent(token=token)
agent.run("What is the soil pH around Kitale, and does it need liming?")

kimeujapheth


╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the soil pH around Kitale, and does it need liming?                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  kitale_location = find_place(name="Kitale")                                                                      
  print(kitale_location)                                                                                           
  kitale_summary = summarize_area(lat=kitale_location["matches"][0]["lat"],                                        
  lon=kitale_location["matches"][0]["lon"], radius_km=3)                                                           
  print(kitale_summary)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'query': 'Kitale', 'matches': [{'name': 'Kitale, Saboti, Trans Nzoia, Kenya', 'lat': 1.01808, 'lon': 35.00024, 
'type': 'town'}]}
{'center': {'lat': 1.01808, 'lon': 35.00024}, 'radius_km': 3.0, 'depth': '0-30 cm', 'properties': {'ph': 
{'summary': 'Soil pH (water) within 3 km: mean 5.4 pH units; area by class: strongly acid 78%, very strongly acid 
12%, moderately acid 10%; confidence: 48% medium, 52% low of the area; 82% of the area is well represented by the 
sampled sites.', 'label': 'Soil pH (water)', 'unit': 'pH units', 'mapped_area_km2': 28.3, 'mean_value': 5.4, 
'min_max_value': [4.8, 5.8], 'dominant_class': 'Strongly acid', 'dominant_class_interpretation': 'Aluminium 
toxicity possible for sensitive crops; liming advised for maize and beans', 'dominant_class_explanation': 
'Exchangeable aluminium may be present and can limit root growth of sensitive crops such as maize and beans; 
phosphorus availability is reduced; raising pH improves fertilizer use efficiency.', 'class_distribution': 
[{'class': 'Strongly acid', 'class_range': '5.1 to <5.6 pH units', 'percent_of_area': 77.8}, {'class': 'Very 
strongly acid', 'class_range': '4.5 to <5.1 pH units', 'percent_of_area': 11.9}, {'class': 'Moderately acid', 
'class_range': '5.6 to <6.1 pH units', 'percent_of_area': 10.3}], 'confidence_percent': {'high': 0.0, 'medium': 
48.0, 'low': 52.0}, 'percent_inside_area_of_applicability': 82.2, 'threshold_source': 'USDA Soil Survey Manual 
(Soil Science Division Staff 2017)'}, 'soc_gkg': {'summary': 'Soil organic carbon within 3 km: mean 16.7 g/kg; area
by class: moderate 77%, high 23%; confidence: 11% high, 87% medium, 2% low of the area; 82% of the area is well 
represented by the sampled sites.', 'label': 'Soil organic carbon', 'unit': 'g/kg', 'mapped_area_km2': 28.3, 
'mean_value': 16.7, 'min_max_value': [12.8, 24.6], 'dominant_class': 'Moderate', 'dominant_class_interpretation': 
'Adequate; maintain with residue retention and organic inputs', 'dominant_class_explanation': 'Enough organic 
matter to support soil structure, water retention and nutrient supply; continued organic inputs are needed to 
prevent decline under continuous cropping.', 'class_distribution': [{'class': 'Moderate', 'class_range': '10 to <18
g/kg', 'percent_of_area': 77.2}, {'class': 'High', 'class_range': '18 to <30 g/kg', 'percent_of_area': 22.8}], 
'confidence_percent': {'high': 10.8, 'medium': 87.3, 'low': 2.0}, 'percent_inside_area_of_applicability': 81.6, 
'threshold_source': 'Hazelton & Murphy (2016) Interpreting Soil Test Results'}}}

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  ph_summary = kitale_summary["properties"]["ph"]["summary"]                                                       
  ph_interpretation = kitale_summary["properties"]["ph"]["dominant_class_interpretation"]                          
  soc_summary = kitale_summary["properties"]["soc_gkg"]["summary"]                                                 
  threshold_source = kitale_summary["properties"]["ph"]["threshold_source"]                                        
  final_answer(f"Soil pH around Kitale, using the first matched location 'Kitale, Saboti, Trans Nzoia, Kenya',     
  needs liming.\n{ph_summary}\n{soc_summary}\n{ph_interpretation}"                                                 
               f"\nPlace used: 'Kitale, Saboti, Trans Nzoia, Kenya'; Depth 0-30 cm; Threshold source:              
  {threshold_source}.")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Soil pH around Kitale, using the first matched location 'Kitale, Saboti, Trans Nzoia, Kenya', needs 
liming.
Soil pH (water) within 3 km: mean 5.4 pH units; area by class: strongly acid 78%, very strongly acid 12%, 
moderately acid 10%; confidence: 48% medium, 52% low of the area; 82% of the area is well represented by the 
sampled sites.
Soil organic carbon within 3 km: mean 16.7 g/kg; area by class: moderate 77%, high 23%; confidence: 11% high, 87% 
medium, 2% low of the area; 82% of the area is well represented by the sampled sites.
Aluminium toxicity possible for sensitive crops; liming advised for maize and beans
Place used: 'Kitale, Saboti, Trans Nzoia, Kenya'; Depth 0-30 cm; Threshold source: USDA Soil Survey Manual (Soil 
Science Division Staff 2017).

[Step 2: Duration 5.09 seconds| Input tokens: 8,908 | Output tokens: 351]

"Soil pH around Kitale, using the first matched location 'Kitale, Saboti, Trans Nzoia, Kenya', needs liming.\nSoil pH (water) within 3 km: mean 5.4 pH units; area by class: strongly acid 78%, very strongly acid 12%, moderately acid 10%; confidence: 48% medium, 52% low of the area; 82% of the area is well represented by the sampled sites.\nSoil organic carbon within 3 km: mean 16.7 g/kg; area by class: moderate 77%, high 23%; confidence: 11% high, 87% medium, 2% low of the area; 82% of the area is well represented by the sampled sites.\nAluminium toxicity possible for sensitive crops; liming advised for maize and beans\nPlace used: 'Kitale, Saboti, Trans Nzoia, Kenya'; Depth 0-30 cm; Threshold source: USDA Soil Survey Manual (Soil Science Division Staff 2017)."

In [7]:
agent.run("Should a farmer near Kakamega town apply lime? Also comment on organic carbon.")

╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ Should a farmer near Kakamega town apply lime? Also comment on organic carbon.                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  kakamega_location = find_place(name="Kakamega town")                                                             
  print(kakamega_location)                                                                                         
  kakamega_soil_summary = summarize_area(lat=kakamega_location["matches"][0]["lat"],                               
  lon=kakamega_location["matches"][0]["lon"], radius_km=3)                                                         
  print(kakamega_soil_summary)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'query': 'Kakamega town', 'matches': [{'name': 'Kakamega, Lurambi, Kakamega, 50100, Kenya', 'lat': 0.2833, 'lon': 
34.75, 'type': 'town'}]}
{'center': {'lat': 0.2833, 'lon': 34.75}, 'radius_km': 3.0, 'depth': '0-30 cm', 'properties': {'ph': {'summary': 
'Soil pH (water) within 3 km: mean 5.0 pH units; area by class: very strongly acid 79%, strongly acid 21%; 
confidence: 98% medium, 2% low of the area; 90% of the area is well represented by the sampled sites.', 'label': 
'Soil pH (water)', 'unit': 'pH units', 'mapped_area_km2': 28.3, 'mean_value': 5.0, 'min_max_value': [4.8, 5.4], 
'dominant_class': 'Very strongly acid', 'dominant_class_interpretation': 'Aluminium toxicity likely and P poorly 
available; liming strongly advised', 'dominant_class_explanation': 'Aluminium toxicity commonly restricts root 
growth of maize and beans; much of the phosphorus (including fertilizer P) is fixed and unavailable; calcium and 
magnesium are often low, so fertilizer responses are poor.', 'class_distribution': [{'class': 'Very strongly acid',
'class_range': '4.5 to <5.1 pH units', 'percent_of_area': 79.3}, {'class': 'Strongly acid', 'class_range': '5.1 to 
<5.6 pH units', 'percent_of_area': 20.7}], 'confidence_percent': {'high': 0.2, 'medium': 98.2, 'low': 1.7}, 
'percent_inside_area_of_applicability': 90.1, 'threshold_source': 'USDA Soil Survey Manual (Soil Science Division 
Staff 2017)'}, 'soc_gkg': {'summary': 'Soil organic carbon within 3 km: mean 19.2 g/kg; area by class: high 93%, 
moderate 7%; confidence: 4% high, 96% medium of the area; 89% of the area is well represented by the sampled 
sites.', 'label': 'Soil organic carbon', 'unit': 'g/kg', 'mapped_area_km2': 28.3, 'mean_value': 19.2, 
'min_max_value': [14.0, 27.2], 'dominant_class': 'High', 'dominant_class_interpretation': 'Good; maintain current 
practices', 'dominant_class_explanation': 'Good structure, water holding and nutrient supply from organic matter, 
supporting active soil biology.', 'class_distribution': [{'class': 'High', 'class_range': '18 to <30 g/kg', 
'percent_of_area': 92.8}, {'class': 'Moderate', 'class_range': '10 to <18 g/kg', 'percent_of_area': 7.2}], 
'confidence_percent': {'high': 4.3, 'medium': 95.6, 'low': 0.0}, 'percent_inside_area_of_applicability': 88.8, 
'threshold_source': 'Hazelton & Murphy (2016) Interpreting Soil Test Results'}}}

Out: None

[Step 1: Duration 3.85 seconds| Input tokens: 3,916 | Output tokens: 151]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(                                                                                                    
      "Liming is strongly advised due to the high levels of acid soil in the area near Kakamega town. "            
      "The soil pH within 3 km has a mean of 5. 0 pH units, with 79% being very strongly acid and 21% strongly     
  acid. "                                                                                                          
      "This indicates potential aluminium toxicity affecting root growth and phosphorus availability,              
  necessitating lime application. "                                                                                
      "Regarding organic carbon, the mean level within 3 km is 19. 2 g/kg, which is classified as high. "          
      "This suggests good soil structure, water holding capacity, and nutrient supply, so maintaining current      
  practices is recommended. "                                                                                      
      "The depth evaluated is 0-30 cm. Source of thresholds: USDA Soil Survey Manual (Soil Science Division Staff  
  2017) for pH and Hazelton & Murphy (2016) Interpreting Soil Test Results for soil organic carbon."               
  )                                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Liming is strongly advised due to the high levels of acid soil in the area near Kakamega town. The 
soil pH within 3 km has a mean of 5. 0 pH units, with 79% being very strongly acid and 21% strongly acid. This 
indicates potential aluminium toxicity affecting root growth and phosphorus availability, necessitating lime 
application. Regarding organic carbon, the mean level within 3 km is 19. 2 g/kg, which is classified as high. This 
suggests good soil structure, water holding capacity, and nutrient supply, so maintaining current practices is 
recommended. The depth evaluated is 0-30 cm. Source of thresholds: USDA Soil Survey Manual (Soil Science Division 
Staff 2017) for pH and Hazelton & Murphy (2016) Interpreting Soil Test Results for soil organic carbon.

[Step 2: Duration 5.29 seconds| Input tokens: 8,889 | Output tokens: 390]

'Liming is strongly advised due to the high levels of acid soil in the area near Kakamega town. The soil pH within 3 km has a mean of 5. 0 pH units, with 79% being very strongly acid and 21% strongly acid. This indicates potential aluminium toxicity affecting root growth and phosphorus availability, necessitating lime application. Regarding organic carbon, the mean level within 3 km is 19. 2 g/kg, which is classified as high. This suggests good soil structure, water holding capacity, and nutrient supply, so maintaining current practices is recommended. The depth evaluated is 0-30 cm. Source of thresholds: USDA Soil Survey Manual (Soil Science Division Staff 2017) for pH and Hazelton & Murphy (2016) Interpreting Soil Test Results for soil organic carbon.'

In [8]:
agent.run("Summarize soil conditions within 5 km of Malava.")

╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ Summarize soil conditions within 5 km of Malava.                                                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  malava_place = find_place(name="Malava")                                                                         
  print("Location details for Malava:", malava_place)                                                              
  malava_coordinates = malava_place["matches"][0]                                                                  
  malava_summary = summarize_area(lat=malava_coordinates["lat"], lon=malava_coordinates["lon"], radius_km=5)       
  print("Soil conditions summary within 5 km of Malava:", malava_summary)                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Location details for Malava: {'query': 'Malava', 'matches': [{'name': 'Malava, Kakamega, Kenya', 'lat': 0.46928, 
'lon': 34.83442, 'type': 'administrative'}, {'name': 'Malava, Chemuche ward, Malava, Kakamega, Kenya', 'lat': 
0.44254, 'lon': 34.85354, 'type': 'town'}]}
Soil conditions summary within 5 km of Malava: {'center': {'lat': 0.46928, 'lon': 34.83442}, 'radius_km': 5.0, 
'depth': '0-30 cm', 'properties': {'ph': {'summary': 'Soil pH (water) within 5 km: mean 5.1 pH units; area by 
class: very strongly acid 64%, strongly acid 36%; confidence: 1% high, 96% medium, 4% low of the area; 95% of the 
area is well represented by the sampled sites.', 'label': 'Soil pH (water)', 'unit': 'pH units', 'mapped_area_km2':
78.5, 'mean_value': 5.1, 'min_max_value': [4.8, 5.5], 'dominant_class': 'Very strongly acid', 
'dominant_class_interpretation': 'Aluminium toxicity likely and P poorly available; liming strongly advised', 
'dominant_class_explanation': 'Aluminium toxicity commonly restricts root growth of maize and beans; much of the 
phosphorus (including fertilizer P) is fixed and unavailable; calcium and magnesium are often low, so fertilizer 
responses are poor.', 'class_distribution': [{'class': 'Very strongly acid', 'class_range': '4.5 to <5.1 pH units',
'percent_of_area': 64.3}, {'class': 'Strongly acid', 'class_range': '5.1 to <5.6 pH units', 'percent_of_area': 
35.7}], 'confidence_percent': {'high': 0.6, 'medium': 95.7, 'low': 3.8}, 'percent_inside_area_of_applicability': 
94.9, 'threshold_source': 'USDA Soil Survey Manual (Soil Science Division Staff 2017)'}, 'soc_gkg': {'summary': 
'Soil organic carbon within 5 km: mean 18.0 g/kg; area by class: high 67%, moderate 33%; confidence: 2% high, 98% 
medium of the area; 93% of the area is well represented by the sampled sites.', 'label': 'Soil organic carbon', 
'unit': 'g/kg', 'mapped_area_km2': 78.5, 'mean_value': 18.0, 'min_max_value': [13.0, 22.0], 'dominant_class': 
'High', 'dominant_class_interpretation': 'Good; maintain current practices', 'dominant_class_explanation': 'Good 
structure, water holding and nutrient supply from organic matter, supporting active soil biology.', 
'class_distribution': [{'class': 'High', 'class_range': '18 to <30 g/kg', 'percent_of_area': 66.6}, {'class': 
'Moderate', 'class_range': '10 to <18 g/kg', 'percent_of_area': 33.4}], 'confidence_percent': {'high': 2.2, 
'medium': 97.8, 'low': 0.0}, 'percent_inside_area_of_applicability': 92.8, 'threshold_source': 'Hazelton & Murphy 
(2016) Interpreting Soil Test Results'}}}

Out: None

[Step 1: Duration 4.03 seconds| Input tokens: 3,912 | Output tokens: 150]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Extracting relevant information from the soil summary                                                          
  location_name = malava_place["matches"][0]["name"]                                                               
  depth_info = malava_summary["depth"]                                                                             
  threshold_source_ph = malava_summary["properties"]["ph"]["threshold_source"]                                     
  threshold_source_soc = malava_summary["properties"]["soc_gkg"]["threshold_source"]                               
                                                                                                                   
  # Soil pH information                                                                                            
  ph_summary = malava_summary["properties"]["ph"]["summary"]                                                       
  ph_interpretation = malava_summary["properties"]["ph"]["dominant_class_interpretation"]                          
  ph_explanation = malava_summary["properties"]["ph"]["dominant_class_explanation"]                                
                                                                                                                   
  # Soil organic carbon information                                                                                
  soc_summary = malava_summary["properties"]["soc_gkg"]["summary"]                                                 
  soc_interpretation = malava_summary["properties"]["soc_gkg"]["dominant_class_interpretation"]                    
  soc_explanation = malava_summary["properties"]["soc_gkg"]["dominant_class_explanation"]                          
                                                                                                                   
  # Constructing the final answer                                                                                  
  final_report = (                                                                                                 
      f"Soil conditions within 5 km of {location_name}:\n\n"                                                       
      f"{ph_summary}\n"                                                                                            
      f"{ph_interpretation}. {ph_explanation}\n\n"                                                                 
      f"{soc_summary}\n"                                                                                           
      f"{soc_interpretation}. {soc_explanation}\n\n"                                                               
      f"Recommendation: Based on the soil pH and organic carbon levels, liming is strongly advised due to          
  aluminum toxicity and poor phosphorus availability. However, the soil organic carbon is high, and current        
  management practices can be maintained.\n\n"                                                                     
      f"Depth: {depth_info}\n"                                                                                     
      f"Threshold Source for pH: {threshold_source_ph}\n"                                                          
      f"Threshold Source for SOC: {threshold_source_soc}"                                                          
  )                                                                                                                
                                                                                                                   
  final_answer(final_report)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Soil conditions within 5 km of Malava, Kakamega, Kenya:

Soil pH (water) within 5 km: mean 5.1 pH units; area by class: very strongly acid 64%, strongly acid 36%; 
confidence: 1% high, 96% medium, 4% low of the area; 95% of the area is well represented by the sampled sites.
Aluminium toxicity likely and P poorly available; liming strongly advised. Aluminium toxicity commonly restricts 
root growth of maize and beans; much of the phosphorus (including fertilizer P) is fixed and unavailable; calcium 
and magnesium are often low, so fertilizer responses are poor.

Soil organic carbon within 5 km: mean 18.0 g/kg; area by class: high 67%, moderate 33%; confidence: 2% high, 98% 
medium of the area; 93% of the area is well represented by the sampled sites.
Good; maintain current practices. Good structure, water holding and nutrient supply from organic matter, supporting
active soil biology.

Recommendation: Based on the soil pH and organic carbon levels, liming is strongly advised due to aluminum toxicity
and poor phosphorus availability. However, the soil organic carbon is high, and current management practices can be
maintained.

Depth: 0-30 cm
Threshold Source for pH: USDA Soil Survey Manual (Soil Science Division Staff 2017)
Threshold Source for SOC: Hazelton & Murphy (2016) Interpreting Soil Test Results

[Step 2: Duration 8.64 seconds| Input tokens: 8,958 | Output tokens: 574]

'Soil conditions within 5 km of Malava, Kakamega, Kenya:\n\nSoil pH (water) within 5 km: mean 5.1 pH units; area by class: very strongly acid 64%, strongly acid 36%; confidence: 1% high, 96% medium, 4% low of the area; 95% of the area is well represented by the sampled sites.\nAluminium toxicity likely and P poorly available; liming strongly advised. Aluminium toxicity commonly restricts root growth of maize and beans; much of the phosphorus (including fertilizer P) is fixed and unavailable; calcium and magnesium are often low, so fertilizer responses are poor.\n\nSoil organic carbon within 5 km: mean 18.0 g/kg; area by class: high 67%, moderate 33%; confidence: 2% high, 98% medium of the area; 93% of the area is well represented by the sampled sites.\nGood; maintain current practices. Good structure, water holding and nutrient supply from organic matter, supporting active soil biology.\n\nRecommendation: Based on the soil pH and organic carbon levels, liming is strongly advised due to

In [9]:
agent.run("What pH values count as strongly acid, and why does it matter for maize?")

╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ What pH values count as strongly acid, and why does it matter for maize?                                        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  class_defns = get_class_definitions(property_name='ph')                                                          
  print(class_defns)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'ph': {'label': 'Soil pH (water)', 'source': 'USDA Soil Survey Manual (Soil Science Division Staff 2017)', 
'classes': [{'class': 'Extremely acid', 'range': '< 4.5 pH units', 'interpretation': 'Aluminium toxicity likely and
severe P fixation; liming essential for most crops', 'explanation': 'Soluble aluminium (and often manganese) 
reaches levels that damage roots, so crops take up little water and nutrients; phosphorus is strongly fixed by iron
and aluminium; calcium and magnesium are usually deficient; legume nitrogen fixation and soil biological activity 
are reduced.'}, {'class': 'Very strongly acid', 'range': '4.5 to <5.1 pH units', 'interpretation': 'Aluminium 
toxicity likely and P poorly available; liming strongly advised', 'explanation': 'Aluminium toxicity commonly 
restricts root growth of maize and beans; much of the phosphorus (including fertilizer P) is fixed and unavailable;
calcium and magnesium are often low, so fertilizer responses are poor.'}, {'class': 'Strongly acid', 'range': '5.1 
to <5.6 pH units', 'interpretation': 'Aluminium toxicity possible for sensitive crops; liming advised for maize and
beans', 'explanation': 'Exchangeable aluminium may be present and can limit root growth of sensitive crops such as 
maize and beans; phosphorus availability is reduced; raising pH improves fertilizer use efficiency.'}, {'class': 
'Moderately acid', 'range': '5.6 to <6.1 pH units', 'interpretation': 'Suitable for most crops; monitor and lime 
only for sensitive crops', 'explanation': 'Aluminium toxicity is rare and most nutrients are adequately available; 
only acid-sensitive crops (e.g. some legumes) are likely to benefit from liming.'}, {'class': 'Slightly acid', 
'range': '6.1 to <6.6 pH units', 'interpretation': 'Near-optimal nutrient availability for most crops', 
'explanation': 'Close to the ideal range for most crops including maize; nutrients are at or near their highest 
availability.'}, {'class': 'Neutral', 'range': '6.6 to <7.4 pH units', 'interpretation': 'Optimal for most crops', 
'explanation': 'Nutrient availability and soil biological activity are generally good for most crops.'}, {'class': 
'Slightly alkaline', 'range': '7.4 to <7.9 pH units', 'interpretation': 'P Zn and Fe availability starts to 
decline', 'explanation': 'As pH rises above neutral, phosphorus, zinc, iron and manganese become less available to 
crops.'}, {'class': 'Alkaline', 'range': '>= 7.9 pH units', 'interpretation': 'Micronutrient deficiencies likely; 
check salinity and sodicity', 'explanation': 'Zinc, iron, manganese and phosphorus deficiencies are common; high pH
can indicate carbonates, salts or sodium that also affect soil structure.'}]}}

Out: None

[Step 1: Duration 2.68 seconds| Input tokens: 3,915 | Output tokens: 91]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  pH_range_strongly_acid = class_defns['ph']['classes'][2]['range']                                                
  pH_effect_strongly_acid = class_defns['ph']['classes'][2]['explanation']                                         
  threshold_source = class_defns['ph']['source']                                                                   
  final_answer(f"Strongly acid pH values range from 5.1 to <5.6 pH units. {pH_effect_strongly_acid}\nPlace used:   
  Not specified, Threshold source: {threshold_source}")                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Strongly acid pH values range from 5.1 to <5.6 pH units. Exchangeable aluminium may be present and 
can limit root growth of sensitive crops such as maize and beans; phosphorus availability is reduced; raising pH 
improves fertilizer use efficiency.
Place used: Not specified, Threshold source: USDA Soil Survey Manual (Soil Science Division Staff 2017)

[Step 2: Duration 4.02 seconds| Input tokens: 8,639 | Output tokens: 254]

'Strongly acid pH values range from 5.1 to <5.6 pH units. Exchangeable aluminium may be present and can limit root growth of sensitive crops such as maize and beans; phosphorus availability is reduced; raising pH improves fertilizer use efficiency.\nPlace used: Not specified, Threshold source: USDA Soil Survey Manual (Soil Science Division Staff 2017)'